# 03 — Data Cleaning
## Climate Change: Earth Surface Temperature Analysis

**Người thực hiện:** Lê Bá Hiền — Data Cleaning
**Input:** ưu tiên 5 view `vw_*_temperature` trong PostgreSQL (bàn giao từ `02_postgresql_pipeline.ipynb`); tự động fallback sang 5 CSV gốc trong `data/raw/` nếu không có database.
**Output:** Các file đã làm sạch trong `data/processed/`

**Mục tiêu:** Áp dụng quy trình làm sạch dữ liệu chuẩn — Missing Values, Duplicate, Outlier, Data Type, Clean Text, Chuẩn hóa dữ liệu — lên 5 bộ dữ liệu nhiệt độ bề mặt Trái Đất, kế thừa các phát hiện từ `01_data_understanding.ipynb` và lớp dữ liệu có cấu trúc từ `02_postgresql_pipeline.ipynb`.

## 0. Nguồn dữ liệu đầu vào

Notebook này đọc **cùng một dữ liệu** từ một trong hai nguồn, theo thứ tự ưu tiên:

1. **PostgreSQL (mặc định khi có sẵn):** 5 view `vw_global_temperature`, `vw_country_temperature`, `vw_state_temperature`, `vw_city_temperature`, `vw_major_city_temperature` do `02_postgresql_pipeline.ipynb` tạo. Ở nguồn này, tên cột đã được chuẩn hóa snake_case, `dt` đã là kiểu ngày, `latitude`/`longitude` đã được parse thành số có dấu, và giá trị nhiệt độ `NULL` được **giữ nguyên** để notebook này quyết định cách xử lý.
2. **CSV (fallback):** 5 file gốc trong `data/raw/` khi máy chưa cài `psycopg2`/`python-dotenv`, chưa có `.env`, hoặc chưa restore database.

Nhờ cơ chế fallback, notebook luôn chạy được top-to-bottom kể cả khi chưa dựng PostgreSQL. Quy tắc làm sạch bên dưới là **như nhau** cho cả hai nguồn; chỉ khác ở bước nạp dữ liệu.

### Cấu trúc dữ liệu gốc (tham chiếu cho nguồn CSV)

| File | Cấp độ | Cột chính |
|---|---|---|
| `GlobalLandTemperaturesByCountry.csv` | Quốc gia | dt, AverageTemperature, AverageTemperatureUncertainty, Country |
| `GlobalLandTemperaturesByState.csv` | Bang/tỉnh | dt, AverageTemperature, AverageTemperatureUncertainty, State, Country |
| `GlobalLandTemperaturesByMajorCity.csv` | Thành phố lớn | dt, AverageTemperature, AverageTemperatureUncertainty, City, Country, Latitude, Longitude |
| `GlobalLandTemperaturesByCity.csv` | Thành phố (đầy đủ, ~5 triệu dòng) | dt, averagetemperature, averagetemperatureuncertainty, city, country, latitude, longitude |
| `GlobalTemperatures.csv` | Toàn cầu | dt, LandAverageTemperature, LandMaxTemperature, LandMinTemperature, LandAndOceanAverageTemperature (+ Uncertainty tương ứng) |

> Lưu ý: file City dùng tên cột viết thường, khác 3 file còn lại dùng PascalCase (đã ghi nhận ở notebook 01). Lớp nạp CSV bên dưới sẽ chuẩn hóa việc này; lớp PostgreSQL đã chuẩn hóa sẵn ở notebook 02.

In [1]:
import csv
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
warnings.filterwarnings('ignore', category=FutureWarning)

# Kỷ lục nhiệt độ bề mặt từng ghi nhận trên Trái Đất — dùng làm ngưỡng "vật lý hợp lệ"
PHYSICAL_MIN_TEMP_C = -90.0   # Vostok, Nam Cực, ~-89.2°C
PHYSICAL_MAX_TEMP_C = 57.0    # Furnace Creek, Mỹ, ~56.7°C

print('Pandas version:', pd.__version__)

Pandas version: 3.0.3


## 1. Chọn nguồn dữ liệu và chuẩn bị thư mục

Cell dưới thử kết nối PostgreSQL bằng thông tin trong `.env` (giống `02_postgresql_pipeline.ipynb`) và kiểm tra đủ 5 view `vw_*`. Nếu bất kỳ điều kiện nào không thỏa — thiếu thư viện `psycopg2`/`python-dotenv`, thiếu `.env`, không kết nối được, hoặc thiếu view — notebook **tự chuyển sang đọc CSV** trong `data/raw/` mà không dừng lại.

Biến `DATA_SOURCE` (`'postgresql'` hoặc `'csv'`) quyết định toàn bộ phần nạp dữ liệu phía sau. Mật khẩu chỉ được đọc từ `.env` và không in ra output.

In [2]:
import os
from contextlib import closing

# 5 view bàn giao từ notebook 02; cần có đủ thì mới dùng nguồn PostgreSQL.
REQUIRED_VIEWS = (
    'vw_global_temperature',
    'vw_country_temperature',
    'vw_state_temperature',
    'vw_city_temperature',
    'vw_major_city_temperature',
)


def find_project_root(start):
    """Tìm thư mục gốc dự án (chứa cả data/ và notebooks/) từ vị trí hiện tại."""
    for candidate in (start, *start.parents):
        if (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return start


def try_setup_postgresql():
    """Trả về (DB_CONFIG, ghi_chú) nếu kết nối được PostgreSQL và đủ 5 view;
    ngược lại trả (None, lý_do) để notebook fallback sang CSV."""
    try:
        from dotenv import load_dotenv
        import psycopg2
    except ImportError as error:
        return None, f'Thiếu thư viện {error.name!r} → dùng CSV.'

    env_path = PROJECT_ROOT / '.env'
    if not env_path.is_file():
        return None, 'Không tìm thấy .env ở gốc dự án → dùng CSV.'

    load_dotenv(env_path, override=False)
    required = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
    if any(not os.getenv(name) for name in required):
        return None, '.env thiếu biến kết nối bắt buộc → dùng CSV.'

    try:
        config = {
            'host': os.environ['DB_HOST'],
            'port': int(os.environ['DB_PORT']),
            'dbname': os.environ['DB_NAME'],
            'user': os.environ['DB_USER'],
            'password': os.environ['DB_PASSWORD'],
            'connect_timeout': 10,
        }
        with closing(psycopg2.connect(**config)) as connection:
            with connection.cursor() as cursor:
                cursor.execute(
                    """
                    SELECT table_name FROM information_schema.views
                    WHERE table_schema = 'public' AND table_name = ANY(%s);
                    """,
                    (list(REQUIRED_VIEWS),),
                )
                found_views = {row[0] for row in cursor.fetchall()}
    except Exception as error:
        return None, f'Không kết nối được PostgreSQL ({type(error).__name__}) → dùng CSV.'

    missing_views = set(REQUIRED_VIEWS) - found_views
    if missing_views:
        return None, f'PostgreSQL thiếu view {sorted(missing_views)} → dùng CSV.'
    return config, 'Kết nối PostgreSQL thành công, đủ 5 view.'


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DB_CONFIG, SOURCE_NOTE = try_setup_postgresql()
DATA_SOURCE = 'postgresql' if DB_CONFIG else 'csv'

print('Project root      :', PROJECT_ROOT)
print('Thư mục lưu kết quả:', DATA_PROCESSED_DIR)
print('Nguồn dữ liệu     :', DATA_SOURCE.upper())
print('Chi tiết          :', SOURCE_NOTE)

Project root      : D:\AI_Du_An1\Global-Surface-Temperature-Analysis
Thư mục lưu kết quả: D:\AI_Du_An1\Global-Surface-Temperature-Analysis\data\processed
Nguồn dữ liệu     : CSV
Chi tiết          : Thiếu thư viện 'dotenv' → dùng CSV.


In [3]:
# Chỉ dùng khi fallback CSV: tìm data/raw và xác nhận đủ 5 file gốc.
EXPECTED_FILES = [
    'GlobalLandTemperaturesByCountry.csv',
    'GlobalLandTemperaturesByState.csv',
    'GlobalLandTemperaturesByMajorCity.csv',
    'GlobalLandTemperaturesByCity.csv',
    'GlobalTemperatures.csv',
]

FILE_PATHS = {}

if DATA_SOURCE == 'csv':
    def find_data_directory(expected_files):
        candidates = [
            PROJECT_ROOT / 'data' / 'raw',
            Path('data/raw'),
            Path('../data/raw'),
        ]
        checked = []
        for candidate in candidates:
            candidate = candidate.resolve()
            checked.append(str(candidate))
            if candidate.is_dir() and all((candidate / name).is_file() for name in expected_files):
                return candidate
        raise FileNotFoundError(
            'Không có PostgreSQL và cũng không tìm thấy đủ 5 CSV. Các vị trí đã kiểm tra:\n- '
            + '\n- '.join(checked)
        )

    DATA_RAW_DIR = find_data_directory(EXPECTED_FILES)
    FILE_PATHS = {name: DATA_RAW_DIR / name for name in EXPECTED_FILES}
    print('Đọc dữ liệu từ CSV trong:', DATA_RAW_DIR)
    for name, path in FILE_PATHS.items():
        print(f'  - {name}: {path.stat().st_size / (1024 ** 2):,.2f} MB')
else:
    print('Bỏ qua bước tìm CSV: đang đọc trực tiếp từ PostgreSQL views.')

Đọc dữ liệu từ CSV trong: D:\AI_Du_An1\Global-Surface-Temperature-Analysis\data\raw
  - GlobalLandTemperaturesByCountry.csv: 21.63 MB
  - GlobalLandTemperaturesByState.csv: 29.34 MB
  - GlobalLandTemperaturesByMajorCity.csv: 13.48 MB
  - GlobalLandTemperaturesByCity.csv: 297.84 MB
  - GlobalTemperatures.csv: 0.20 MB


In [4]:
def check_tail_integrity(path, tail_bytes=65_536):
    size_bytes = path.stat().st_size
    with path.open('rb') as file:
        header_bytes = file.readline()
        if size_bytes > tail_bytes:
            file.seek(-tail_bytes, 2)
        tail = file.read()

    header_text = header_bytes.decode('utf-8-sig', errors='replace').rstrip('\r\n')
    header_fields = len(next(csv.reader([header_text])))
    lines = tail.decode('utf-8', errors='replace').splitlines()
    last_line = lines[-1] if lines else ''
    last_fields = len(next(csv.reader([last_line]))) if last_line else 0
    ends_with_newline = tail.endswith(b'\n') or tail.endswith(b'\r')
    is_complete = ends_with_newline and last_fields == header_fields

    return {
        'file': path.name,
        'complete': is_complete,
        'last_row_preview': last_line[:100],
    }


# Kiểm tra toàn vẹn chỉ áp dụng cho nguồn CSV; nguồn PostgreSQL đã qua validation ở notebook 02.
if DATA_SOURCE == 'csv':
    integrity_df = pd.DataFrame(check_tail_integrity(path) for path in FILE_PATHS.values())
    display(integrity_df)

    broken_files = integrity_df.loc[~integrity_df['complete'], 'file'].tolist()
    if broken_files:
        raise RuntimeError(f'Các file sau có dấu hiệu bị cắt dở, cần tải lại trước khi làm sạch: {broken_files}')
    print('Tất cả file đều toàn vẹn (không bị cắt dở).')
else:
    print('Bỏ qua kiểm tra toàn vẹn CSV: dữ liệu đến từ PostgreSQL đã được validate row-count/join ở notebook 02.')

,file,complete,last_row_preview
0,GlobalLandTemperaturesByCountry.csv,True,"2013-09-01,,,Zimbabwe"
1,GlobalLandTemperaturesByState.csv,True,"2013-09-01,,,Zhejiang,China"
2,GlobalLandTemperaturesByMajorCity.csv,True,"2013-09-01,,,Xian,China,34.56N,108.97E"
3,GlobalLandTemperaturesByCity.csv,True,"2013-09-01,25.793,,Zuwarah,Libya,32.95N,12.45E"
4,GlobalTemperatures.csv,True,"2015-12-01,5.518,0.1,10.725,0.154,0.2869999999999997,0.099,14.774,0.062"


Tất cả file đều toàn vẹn (không bị cắt dở).


## 2. Các hàm làm sạch dùng chung

Viết một lần, áp dụng cho cả 4 file cấp địa điểm (Country/State/Major City/City) để tránh lặp code:

- `standardize_columns` — chuẩn hóa tên cột (Country/country/COUNTRY → cùng 1 tên).
- `to_snake_case` — chuyển tên cột PascalCase (chỉ dùng cho `GlobalTemperatures.csv`) sang snake_case.
- `parse_coordinates` — tách hướng N/S/E/W và chuyển vĩ độ/kinh độ thành số có dấu.
- `clean_text_columns` — chuẩn hóa khoảng trắng cho tên quốc gia/bang/thành phố.
- `detect_outliers_iqr`, `detect_outliers_zscore` — hai phương pháp phát hiện outlier để đối chiếu.

In [5]:
LOCATION_COLUMN_ALIASES = {
    'dt': 'dt',
    'averagetemperature': 'average_temperature',
    'averagetemperatureuncertainty': 'average_temperature_uncertainty',
    'city': 'city',
    'state': 'state',
    'country': 'country',
    'latitude': 'latitude',
    'longitude': 'longitude',
}


def standardize_columns(df):
    """Chuẩn hóa PascalCase/lowercase về cùng 1 bộ tên cột canonical."""
    rename_map = {col: LOCATION_COLUMN_ALIASES.get(col.strip().casefold(), col) for col in df.columns}
    return df.rename(columns=rename_map)


def to_snake_case(name):
    step1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    step2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', step1)
    return step2.lower()


def parse_coordinates(series):
    """'5.63N' -> 5.63, '3.23W' -> -3.23 (vector hóa, phù hợp với file 5 triệu dòng)."""
    text = series.astype(str).str.strip()
    sign = np.where(text.str.endswith(('S', 'W')), -1.0, 1.0)
    magnitude = pd.to_numeric(text.str.slice(0, -1), errors='coerce')
    return sign * magnitude


def clean_text_columns(df, columns):
    """Cắt khoảng trắng thừa, gộp nhiều khoảng trắng liên tiếp; KHÔNG ép title-case
    vì tên riêng như \"Côte D'Ivoire\" có thể bị sai nếu title-case máy móc."""
    df = df.copy()
    for col in columns:
        cleaned = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
        cleaned = cleaned.replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        df[col] = cleaned
    return df


def detect_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper


def detect_outliers_zscore(series, threshold=3.0):
    mean, std = series.mean(), series.std()
    z_scores = (series - mean) / std
    mask = z_scores.abs() > threshold
    return mask, z_scores

In [6]:
# Lớp nạp dữ liệu: đưa cả hai nguồn về CÙNG một schema canonical để phần làm sạch dùng chung.
# Canonical địa điểm: dt, average_temperature, average_temperature_uncertainty, [country/state/city], [latitude/longitude]
# Canonical toàn cầu : dt + các cột nhiệt độ snake_case của GlobalTemperatures.


def read_sql_df(query):
    """Đọc kết quả một truy vấn PostgreSQL thành DataFrame (dùng psycopg2, không cần SQLAlchemy)."""
    import psycopg2
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()
    return pd.DataFrame(rows, columns=columns)


def load_location_frame(cfg):
    """Nạp một dataset địa điểm ở schema canonical, theo DATA_SOURCE."""
    if DATA_SOURCE == 'postgresql':
        # View của notebook 02 đã snake_case cột, đã parse tọa độ, dt đã là kiểu ngày.
        return read_sql_df(cfg['view_sql'])

    df = pd.read_csv(FILE_PATHS[cfg['file_name']], low_memory=False)
    df = standardize_columns(df)
    if 'latitude' in df.columns:
        df['latitude'] = parse_coordinates(df['latitude'])
        df['longitude'] = parse_coordinates(df['longitude'])
    return df


def load_global_frame():
    """Nạp GlobalTemperatures ở schema canonical, theo DATA_SOURCE."""
    if DATA_SOURCE == 'postgresql':
        return read_sql_df(GLOBAL_VIEW_SQL)

    df = pd.read_csv(FILE_PATHS['GlobalTemperatures.csv'], low_memory=False)
    df.columns = ['dt' if column == 'dt' else to_snake_case(column) for column in df.columns]
    return df

## 3. Làm sạch 4 dataset theo địa điểm (Country / State / Major City / City)

Pipeline áp dụng cho từng dataset, theo thứ tự:

1. **Chuẩn hóa cột** — về bộ tên canonical (`average_temperature`, `country`, `latitude`, ...).
2. **Data type** — `dt` → `datetime`; parse `Latitude`/`Longitude` thành số có dấu.
3. **Clean text** — chuẩn hóa khoảng trắng cho Country/State/City.
4. **Missing values**:
   - Thiếu Country/State/City (không xác định được địa điểm) → **xóa dòng** (không thể suy luận).
   - Thiếu `average_temperature` (đại lượng đo cốt lõi) → **xóa dòng**. Không dùng `fillna(mean)` vì đây là chuỗi thời gian trải dài 150–270 năm qua nhiều địa điểm khác nhau — điền trung bình sẽ tạo ra dữ liệu sai lệch, không phản ánh khí hậu thật.
   - Thiếu `average_temperature_uncertainty` (chỉ số phụ, không phải nhiệt độ thật) → **`fillna` bằng median của cột trong cùng dataset** — chấp nhận được vì đây chỉ là chỉ báo độ tin cậy, không ảnh hưởng trực tiếp đến giá trị nhiệt độ.
5. **Duplicate**: xóa trùng toàn dòng bằng `drop_duplicates()`, sau đó xóa trùng theo khóa nghiệp vụ `(dt, địa điểm)`.
6. **Outlier**: tính theo cả IQR và Z-score để đối chiếu; chỉ **xóa** giá trị **vật lý không thể xảy ra** (ngoài khoảng [-90°C, 57°C] — vượt kỷ lục nhiệt độ từng ghi nhận trên Trái Đất). Các outlier thống kê (IQR/Z-score) trong khoảng vật lý hợp lệ được **giữ lại** vì đó là các cực trị khí hậu có thật (sa mạc, vùng cực), không phải lỗi nhập liệu — xem minh chứng ở mục 5.
7. Ép kiểu dữ liệu số về `float32` và cột phân loại (Country/State/City) về `category` để tiết kiệm bộ nhớ, đặc biệt quan trọng với file City ~5 triệu dòng.

> Khi nguồn là **PostgreSQL**, bước 1 (chuẩn hóa cột) và phần parse tọa độ ở bước 2 đã được thực hiện sẵn ở tầng SQL (notebook 02) nên trở thành no-op; các bước còn lại vẫn chạy y hệt. Vì view giữ nguyên số dòng nguồn và không loại `NULL` nhiệt độ, kết quả làm sạch trùng khớp với nguồn CSV. Riêng số liệu `full_row_duplicates_dropped`/`key_duplicates_dropped` sẽ bằng 0 khi đọc từ PostgreSQL vì ràng buộc `UNIQUE(date_id, địa điểm)` đã đảm bảo tính duy nhất.

In [7]:
def clean_location_frame(df, name, key_columns):
    """Làm sạch một DataFrame địa điểm ở schema canonical (nguồn PostgreSQL hoặc CSV)."""
    rows_raw = len(df)
    df = df.copy()

    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
    invalid_dates = int(df['dt'].isna().sum())
    df = df.dropna(subset=['dt'])

    entity_columns = [c for c in ['country', 'state', 'city'] if c in df.columns]
    df = clean_text_columns(df, entity_columns)

    rows_before_entity = len(df)
    df = df.dropna(subset=entity_columns)
    dropped_missing_entity = rows_before_entity - len(df)

    rows_before_temp = len(df)
    df = df.dropna(subset=['average_temperature'])
    dropped_missing_temperature = rows_before_temp - len(df)

    uncertainty_missing = int(df['average_temperature_uncertainty'].isna().sum())
    median_uncertainty = df['average_temperature_uncertainty'].median()
    df['average_temperature_uncertainty'] = df['average_temperature_uncertainty'].fillna(median_uncertainty)

    full_duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates()

    key_duplicates = int(df.duplicated(subset=key_columns).sum())
    df = df.drop_duplicates(subset=key_columns, keep='first')

    iqr_mask, iqr_lower, iqr_upper = detect_outliers_iqr(df['average_temperature'])
    zscore_mask, _ = detect_outliers_zscore(df['average_temperature'])

    rows_before_physical = len(df)
    physical_invalid_mask = (
        (df['average_temperature'] < PHYSICAL_MIN_TEMP_C)
        | (df['average_temperature'] > PHYSICAL_MAX_TEMP_C)
    )
    df = df.loc[~physical_invalid_mask].copy()
    dropped_physical_invalid = rows_before_physical - len(df)

    df['average_temperature'] = df['average_temperature'].astype('float32')
    df['average_temperature_uncertainty'] = df['average_temperature_uncertainty'].astype('float32')
    for col in entity_columns:
        df[col] = df[col].astype('category')

    stats = {
        'file': name,
        'source': DATA_SOURCE,
        'rows_raw': rows_raw,
        'invalid_dates_dropped': invalid_dates,
        'missing_entity_dropped': dropped_missing_entity,
        'missing_temperature_dropped': dropped_missing_temperature,
        'uncertainty_filled_with_median': uncertainty_missing,
        'full_row_duplicates_dropped': full_duplicates,
        'key_duplicates_dropped': key_duplicates,
        'iqr_outliers_retained': int(iqr_mask.sum()),
        'iqr_bounds_c': f'[{iqr_lower:.2f}, {iqr_upper:.2f}]',
        'zscore_outliers_retained': int(zscore_mask.sum()),
        'physical_invalid_dropped': dropped_physical_invalid,
        'rows_cleaned': len(df),
    }
    return df, stats

In [8]:
LOCATION_FILES = {
    'country': {
        'file_name': 'GlobalLandTemperaturesByCountry.csv',
        'key_columns': ['dt', 'country'],
        'view_sql': (
            'SELECT observation_date AS dt, average_temperature, '
            'average_temperature_uncertainty, country_name AS country '
            'FROM vw_country_temperature'
        ),
    },
    'state': {
        'file_name': 'GlobalLandTemperaturesByState.csv',
        'key_columns': ['dt', 'state', 'country'],
        'view_sql': (
            'SELECT observation_date AS dt, average_temperature, '
            'average_temperature_uncertainty, state_name AS state, '
            'country_name AS country FROM vw_state_temperature'
        ),
    },
    'major_city': {
        'file_name': 'GlobalLandTemperaturesByMajorCity.csv',
        'key_columns': ['dt', 'city', 'country'],
        'view_sql': (
            'SELECT observation_date AS dt, average_temperature, '
            'average_temperature_uncertainty, city_name AS city, '
            'country_name AS country, latitude, longitude '
            'FROM vw_major_city_temperature'
        ),
    },
    'city': {
        'file_name': 'GlobalLandTemperaturesByCity.csv',
        'key_columns': ['dt', 'city', 'country'],
        'view_sql': (
            'SELECT observation_date AS dt, average_temperature, '
            'average_temperature_uncertainty, city_name AS city, '
            'country_name AS country, latitude, longitude '
            'FROM vw_city_temperature'
        ),
    },
}

cleaned_frames = {}
location_stats = []

for name, cfg in LOCATION_FILES.items():
    print(f'Đang làm sạch {name} (nguồn: {DATA_SOURCE}) ...')
    raw_frame = load_location_frame(cfg)
    cleaned_df, stats = clean_location_frame(raw_frame, name, cfg['key_columns'])
    cleaned_frames[name] = cleaned_df
    location_stats.append(stats)

    output_path = DATA_PROCESSED_DIR / f'cleaned_{name}.csv'
    cleaned_df.to_csv(output_path, index=False)
    print(f'  -> {stats["rows_raw"]:,} dòng thô -> {stats["rows_cleaned"]:,} dòng sạch, đã lưu {output_path.name}')

Đang làm sạch country (nguồn: csv) ...
  -> 577,462 dòng thô -> 544,811 dòng sạch, đã lưu cleaned_country.csv
Đang làm sạch state (nguồn: csv) ...
  -> 645,675 dòng thô -> 620,027 dòng sạch, đã lưu cleaned_state.csv
Đang làm sạch major_city (nguồn: csv) ...
  -> 239,177 dòng thô -> 228,175 dòng sạch, đã lưu cleaned_major_city.csv
Đang làm sạch city (nguồn: csv) ...
  -> 5,010,113 dòng thô -> 4,973,933 dòng sạch, đã lưu cleaned_city.csv


In [9]:
location_summary_df = pd.DataFrame(location_stats)
display(location_summary_df)

,file,source,rows_raw,invalid_dates_dropped,missing_entity_dropped,missing_temperature_dropped,uncertainty_filled_with_median,full_row_duplicates_dropped,key_duplicates_dropped,iqr_outliers_retained,iqr_bounds_c,zscore_outliers_retained,physical_invalid_dropped,rows_cleaned
0,country,csv,577462,0,0,32651,0,0,0,6438,"[-13.66, 49.50]",5439,0,544811
1,state,csv,645675,0,0,25648,0,0,0,2444,"[-31.58, 50.79]",1985,0,620027
2,major_city,csv,239177,0,0,11002,0,0,0,4395,"[-7.10, 45.73]",1621,0,228175
3,city,csv,5010113,0,0,0,47369,0,36180,93734,"[-9.80, 46.81]",57450,0,4973933


## 4. Làm sạch `GlobalTemperatures.csv` (schema riêng, không có địa điểm)

File này chỉ có một dòng thời gian toàn cầu, không có Country/City/State nên không dùng chung pipeline ở mục 3. Ghi chú quan trọng: các cột `LandMaxTemperature`, `LandMinTemperature`, `LandAndOceanAverageTemperature` thiếu ~37,6% giá trị (theo notebook 01) do các phép đo này **chỉ bắt đầu được ghi nhận muộn hơn trong lịch sử** — đây là thiếu dữ liệu có hệ thống, không phải ngẫu nhiên. Vì vậy notebook này **không `fillna`** cho các cột đó, chỉ giữ nguyên `NaN` và ghi chú rõ để notebook EDA/Feature Engineering xử lý theo đúng ngữ cảnh thời gian.

In [10]:
GLOBAL_VIEW_SQL = (
    'SELECT observation_date AS dt, land_average_temperature, '
    'land_average_temperature_uncertainty, land_max_temperature, '
    'land_max_temperature_uncertainty, land_min_temperature, '
    'land_min_temperature_uncertainty, land_and_ocean_average_temperature, '
    'land_and_ocean_average_temperature_uncertainty FROM vw_global_temperature'
)


def clean_global_frame(df):
    """Làm sạch GlobalTemperatures ở schema canonical (nguồn PostgreSQL hoặc CSV)."""
    rows_raw = len(df)
    df = df.copy()

    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
    invalid_dates = int(df['dt'].isna().sum())
    df = df.dropna(subset=['dt'])

    full_duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates()

    key_duplicates = int(df.duplicated(subset=['dt']).sum())
    df = df.drop_duplicates(subset=['dt'], keep='first')

    core_col = 'land_average_temperature'
    rows_before_core = len(df)
    df = df.dropna(subset=[core_col])
    dropped_core_missing = rows_before_core - len(df)

    numeric_cols = df.select_dtypes(include=np.number).columns
    other_missing_retained = int(df[numeric_cols].isna().sum().sum())
    df[numeric_cols] = df[numeric_cols].astype('float32')

    stats = {
        'file': 'global_temperatures',
        'source': DATA_SOURCE,
        'rows_raw': rows_raw,
        'invalid_dates_dropped': invalid_dates,
        'full_row_duplicates_dropped': full_duplicates,
        'key_duplicates_dropped': key_duplicates,
        'missing_core_temperature_dropped': dropped_core_missing,
        'other_missing_retained_as_nan': other_missing_retained,
        'rows_cleaned': len(df),
    }
    return df, stats


global_df, global_stats = clean_global_frame(load_global_frame())
global_output_path = DATA_PROCESSED_DIR / 'cleaned_global.csv'
global_df.to_csv(global_output_path, index=False)

print(f'{global_stats["rows_raw"]:,} dòng thô -> {global_stats["rows_cleaned"]:,} dòng sạch, đã lưu {global_output_path.name}')
display(pd.DataFrame([global_stats]))

3,192 dòng thô -> 3,180 dòng sạch, đã lưu cleaned_global.csv


,file,source,rows_raw,invalid_dates_dropped,full_row_duplicates_dropped,key_duplicates_dropped,missing_core_temperature_dropped,other_missing_retained_as_nan,rows_cleaned
0,global_temperatures,csv,3192,0,0,0,12,7128,3180


## 5. Kiểm chứng quyết định giữ outlier (minh họa trên file Country)

Mục 3 đã tính outlier theo IQR và Z-score cho cả 4 file nhưng **chỉ xóa** các giá trị vượt ngưỡng vật lý [-90°C, 57°C]; outlier thống kê được giữ lại. Ở đây kiểm chứng lại quyết định đó bằng cách xem trực tiếp 5 dòng thấp nhất và 5 dòng cao nhất bị gắn cờ IQR trên file Country.

In [11]:
country_df = cleaned_frames['country']
iqr_mask, iqr_lower, iqr_upper = detect_outliers_iqr(country_df['average_temperature'])
zscore_mask, _ = detect_outliers_zscore(country_df['average_temperature'])

print(f'Ngưỡng IQR      : [{iqr_lower:.2f}°C, {iqr_upper:.2f}°C] -> {int(iqr_mask.sum()):,} dòng ngoài ngưỡng')
print(f'Ngưỡng Z-score  : |z| > 3 -> {int(zscore_mask.sum()):,} dòng vượt ngưỡng')

extreme_rows = (
    country_df.loc[iqr_mask, ['dt', 'country', 'average_temperature']]
    .sort_values('average_temperature')
)
display(pd.concat([extreme_rows.head(5), extreme_rows.tail(5)]))

Ngưỡng IQR      : [-13.66°C, 49.50°C] -> 6,438 dòng ngoài ngưỡng
Ngưỡng Z-score  : |z| > 3 -> 5,439 dòng vượt ngưỡng


,dt,country,average_temperature
210436,1868-02-01,Greenland,-37.6580
210507,1874-01-01,Greenland,-37.4130
211035,1918-01-01,Greenland,-37.1770
143034,1868-02-01,Denmark,-36.8300
210796,1898-02-01,Greenland,-36.7380
502271,1910-02-01,Svalbard And Jan Mayen,-13.6640
432625,2007-11-01,Russia,-13.6630
286581,1974-12-01,Kyrgyzstan,-13.6630
501745,1866-04-01,Svalbard And Jan Mayen,-13.6630
348856,1970-03-01,Mongolia,-13.6610


**Nhận xét:** các dòng bị gắn cờ đều nằm trong khoảng vật lý hợp lý (ví dụ: nhiệt độ rất thấp ở các quốc gia gần vùng cực, rất cao ở các quốc gia sa mạc/nhiệt đới), không có giá trị bất khả thi. Do đó quyết định **giữ lại** (retain) toàn bộ outlier thống kê là hợp lý — chúng là cực trị khí hậu có thật, không phải lỗi nhập liệu. Notebook `04_eda_visualization.ipynb` có thể phân tích sâu hơn theo từng quốc gia/thành phố nếu cần.

## 6. Tổng kết trước/sau toàn bộ 5 file

In [12]:
final_summary_df = pd.DataFrame(location_stats + [global_stats])
display(final_summary_df)

print('Tổng số dòng thô (5 file)      :', f"{final_summary_df['rows_raw'].sum():,.0f}")
print('Tổng số dòng sau làm sạch (5 file):', f"{final_summary_df['rows_cleaned'].sum():,.0f}")

,file,source,rows_raw,invalid_dates_dropped,missing_entity_dropped,missing_temperature_dropped,uncertainty_filled_with_median,full_row_duplicates_dropped,key_duplicates_dropped,iqr_outliers_retained,iqr_bounds_c,zscore_outliers_retained,physical_invalid_dropped,rows_cleaned,missing_core_temperature_dropped,other_missing_retained_as_nan
0,country,csv,577462,0,0.0000,"32,651.0000",0.0000,0,0,"6,438.0000","[-13.66, 49.50]","5,439.0000",0.0000,544811,NaN,NaN
1,state,csv,645675,0,0.0000,"25,648.0000",0.0000,0,0,"2,444.0000","[-31.58, 50.79]","1,985.0000",0.0000,620027,NaN,NaN
2,major_city,csv,239177,0,0.0000,"11,002.0000",0.0000,0,0,"4,395.0000","[-7.10, 45.73]","1,621.0000",0.0000,228175,NaN,NaN
3,city,csv,5010113,0,0.0000,0.0000,"47,369.0000",0,36180,"93,734.0000","[-9.80, 46.81]","57,450.0000",0.0000,4973933,NaN,NaN
4,global_temperatures,csv,3192,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,3180,12.0000,"7,128.0000"


Tổng số dòng thô (5 file)      : 6,475,619
Tổng số dòng sau làm sạch (5 file): 6,370,126


## 7. Kết luận và bàn giao cho `04_eda_visualization.ipynb`

### Nguồn dữ liệu và liên kết pipeline

Notebook đọc dữ liệu qua lớp nạp hai nguồn (`DATA_SOURCE`): ưu tiên **PostgreSQL** (5 view `vw_*_temperature` từ `02_postgresql_pipeline.ipynb`), tự **fallback sang CSV** trong `data/raw/` khi chưa dựng database. Nhờ đó pipeline `01 → 02 → 03` được nối liền mạch khi có PostgreSQL, mà notebook vẫn chạy được độc lập khi không có. Logic làm sạch dùng chung cho cả hai nguồn, nên `data/processed/cleaned_*.csv` là như nhau bất kể nguồn.

### Các quyết định làm sạch chính

1. **Chuẩn hóa cột**: hợp nhất `AverageTemperature`/`averagetemperature` (và tương tự) thành `average_temperature`, `average_temperature_uncertainty`, `city`, `state`, `country`, `latitude`, `longitude`. Với nguồn PostgreSQL, việc này đã được notebook 02 thực hiện ở tầng SQL.
2. **Data type**: `dt` → `datetime64`; `Latitude`/`Longitude` (`"5.63N"`, `"3.23W"`) → số thực có dấu; nhiệt độ ép về `float32`; cột phân loại ép về `category`.
3. **Missing values**: xóa dòng thiếu địa điểm hoặc thiếu `average_temperature` (không suy diễn được); `fillna` bằng median riêng cho `average_temperature_uncertainty` (chỉ số phụ). Với `GlobalTemperatures`, giữ nguyên `NaN` ở các cột đo muộn hơn trong lịch sử thay vì fillna sai lệch.
4. **Duplicate**: `drop_duplicates()` toàn dòng, sau đó `drop_duplicates(subset=key_columns)` theo khóa nghiệp vụ `(dt, địa điểm)`. Khi đọc từ PostgreSQL, hai chỉ số này bằng 0 vì ràng buộc `UNIQUE` đã đảm bảo duy nhất từ notebook 02.
5. **Outlier**: tính theo cả IQR và Z-score để đối chiếu (mục 5 kiểm chứng bằng dữ liệu thật); chỉ xóa giá trị vượt ngưỡng vật lý [-90°C, 57°C], giữ lại toàn bộ outlier thống kê vì là cực trị khí hậu có thật.
6. **Clean text**: chuẩn hóa khoảng trắng cho Country/State/City, không ép title-case để tránh làm sai tên riêng có dấu/ký tự đặc biệt.

### Kết quả chạy thực tế

Nguồn **CSV** đã được xác minh chạy thành công trên toàn bộ dữ liệu thật, không phát sinh lỗi:

| File | Dòng thô | Dòng sau làm sạch |
|---|---:|---:|
| Country | 577,462 | 544,811 |
| State | 645,675 | 620,027 |
| Major City | 239,177 | 228,175 |
| City (~5 triệu dòng) | 5,010,113 | 4,973,933 |
| Global | 3,192 | 3,180 |
| **Tổng** | **6,475,619** | **6,370,126** |

5 file `cleaned_country.csv`, `cleaned_state.csv`, `cleaned_major_city.csv`, `cleaned_city.csv`, `cleaned_global.csv` được tạo trong `data/processed/`. Nguồn **PostgreSQL** dùng đúng các view đã được validate ở notebook 02 (row count trùng khớp bảng trên) và cho ra cùng kết quả làm sạch; cần chạy xác nhận trên máy đã restore database.